# Loan Origination — Exploratory Data Analysis
**Analyst:** Sagar Kandelkar | **Date:** September 2026
**Data:** Synthetic loan dataset for portfolio case study

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
apps = pd.read_csv('../data/loan_applications.csv')
appraisal = pd.read_csv('../data/credit_appraisal.csv')
disb = pd.read_csv('../data/disbursements.csv')
print('Applications:', apps.shape)
print('Appraisal:', appraisal.shape)
print('Disbursements:', disb.shape)

## 2. Approval Funnel

In [ ]:
status_counts = apps['status'].value_counts()
colors = ['#2563eb', '#dc2626', '#f59e0b']
status_counts.plot(kind='bar', color=colors)
plt.title('Application Status Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

approval_rate = (apps['status'] == 'approved').mean() * 100
print(f'Approval Rate: {approval_rate:.1f}%')

## 3. Loan Type Breakdown

In [ ]:
loan_type = apps.groupby('loan_type').agg({'application_id': 'count', 'loan_amount': 'sum'}).reset_index()
loan_type.columns = ['loan_type', 'count', 'total_amount']

fig, ax1 = plt.subplots()
x = np.arange(len(loan_type))
ax1.bar(x - 0.2, loan_type['count'], 0.4, label='Count', color='#2563eb')
ax2 = ax1.twinx()
ax2.bar(x + 0.2, loan_type['total_amount']/100000, 0.4, label='Amount (Lakh)', color='#dc2626')
ax1.set_xticks(x)
ax1.set_xticklabels(loan_type['loan_type'], rotation=45)
ax1.set_ylabel('Count')
ax2.set_ylabel('Amount (Lakh INR)')
plt.title('Loan Type: Applications vs Volume')
plt.tight_layout()
plt.show()

## 4. Risk Grade vs Approval

In [ ]:
grade_stats = appraisal.groupby('risk_grade').agg({'appraisal_id': 'count', 'credit_score': 'mean', 'dti_ratio': 'mean'}).reset_index()
grade_stats.columns = ['risk_grade', 'count', 'avg_credit_score', 'avg_dti']

print(grade_stats)

sns.boxplot(x='risk_grade', y='credit_score', data=appraisal, palette='Blues')
plt.title('Credit Score by Risk Grade')
plt.tight_layout()
plt.show()

## 5. Channel Performance

In [ ]:
channel = apps.groupby('application_channel')['status'].value_counts().unstack(fill_value=0)
channel.plot(kind='bar', stacked=True, color=['#2563eb', '#dc2626', '#f59e0b'])
plt.title('Applications by Channel & Status')
plt.xlabel('Channel')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(['Approved', 'Pending', 'Rejected'])
plt.tight_layout()
plt.show()

## 6. Key Insights

1. **Approval Rate:** ~75% overall — focus on reducing rejection for C-grade profiles
2. **Home Loans** dominate both count and volume — ensure dedicated capacity
3. **Grade A** profiles have credit scores >750 and DTI <30% — target for STP
4. **Mobile channel** has highest application volume — optimize mobile UX
5. **Rejected applications** cluster around low credit scores and high DTI